# 06A - Creating a Pipeline — Azure ML SDK v2 Version

This notebook is a modern SDK v2 replacement for the old SDK v1 pipeline lab.

The old lab used deprecated/legacy patterns such as `azureml.core`, `PipelineData`, `EstimatorStep`, `PythonScriptStep`, `RunConfiguration`, and `RunDetails`. This version uses the current Azure ML SDK v2 style:

- `MLClient` for workspace access
- `Data` assets instead of v1 `Dataset`
- `command` components instead of `EstimatorStep` / `PythonScriptStep`
- `dsl.pipeline` for pipeline orchestration
- `Output` objects instead of `PipelineData`
- `Model` registration from the completed pipeline output

> Important: Run this notebook inside Azure Machine Learning Studio or an environment where you are authenticated to your Azure subscription.

## 1. Connect to the Azure ML workspace

This uses `MLClient.from_config()`, which reads the workspace configuration available in Azure ML Studio notebooks.

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print("Connected to workspace:", ml_client.workspace_name)
print("Resource group:", ml_client.resource_group_name)
print("Subscription:", ml_client.subscription_id)

## 2. Check that local data files exist

This lab expects the diabetes CSV files to be available in the local `./data` folder.

In [ ]:
from pathlib import Path

DATA_FOLDER = Path("./data")
expected_files = [DATA_FOLDER / "diabetes.csv", DATA_FOLDER / "diabetes2.csv"]

print("Current folder:", Path.cwd())
print("Data folder exists:", DATA_FOLDER.exists())

if DATA_FOLDER.exists():
    print("Files in ./data:")
    for item in DATA_FOLDER.iterdir():
        print(" -", item)

missing = [str(p) for p in expected_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required data files: " + ", ".join(missing) +
        "
Upload diabetes.csv and diabetes2.csv into a folder named data beside this notebook."
    )

## 3. Create or update a Data asset

SDK v1 used `Dataset.Tabular`. In SDK v2, we usually register a `Data` asset. Here we upload the entire `./data` folder as a `uri_folder` asset.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from datetime import datetime

DATA_ASSET_NAME = "diabetes-dataset"
data_version = datetime.now().strftime("%Y%m%d%H%M%S")

diabetes_data = Data(
    name=DATA_ASSET_NAME,
    version=data_version,
    type=AssetTypes.URI_FOLDER,
    path=str(DATA_FOLDER),
    description="Diabetes CSV files uploaded from local ./data folder for SDK v2 pipeline lab",
    tags={"format": "CSV", "lab": "06A pipeline SDK v2"}
)

diabetes_data = ml_client.data.create_or_update(diabetes_data)

print("Data asset created/updated")
print("Name:", diabetes_data.name)
print("Version:", diabetes_data.version)
print("Path:", diabetes_data.path)

## 4. Create the pipeline source folder

This folder will contain the Python script used by the pipeline component.

In [ ]:
import os
from pathlib import Path

experiment_folder = Path("diabetes_pipeline_v2")
experiment_folder.mkdir(exist_ok=True)

print("Pipeline source folder:", experiment_folder.resolve())

## 5. Create the training script

This script receives a data-folder input and a model-folder output from the pipeline. It trains a decision-tree model, writes the model to the output folder, and saves the ROC chart there too.

Notice the important fix: we do **not** use `mlflow.log_figure()`. That method can fail in Azure ML when MLflow and Azure ML artifact plugins are mismatched. Instead, the chart is saved directly into the component output folder.

In [ ]:
%%writefile $experiment_folder/train_diabetes.py
import argparse
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


def load_csv_folder(folder_path: str) -> pd.DataFrame:
    folder = Path(folder_path)
    csv_files = sorted(folder.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files were found in {folder}. "
            "Check that the input Data asset points to a folder containing diabetes.csv and diabetes2.csv."
        )

    print("CSV files found:")
    for file in csv_files:
        print(" -", file)

    frames = [pd.read_csv(file) for file in csv_files]
    return pd.concat(frames, ignore_index=True)


parser = argparse.ArgumentParser()
parser.add_argument("--training_data", type=str, required=True, help="Input folder containing diabetes CSV files")
parser.add_argument("--output_model", type=str, required=True, help="Output folder for model and artifacts")
args = parser.parse_args()

print("Training data path:", args.training_data)
print("Output model path:", args.output_model)

output_folder = Path(args.output_model)
output_folder.mkdir(parents=True, exist_ok=True)

print("Loading data...")
diabetes = load_csv_folder(args.training_data)
print("Rows loaded:", len(diabetes))
print("Columns:", list(diabetes.columns))

feature_columns = [
    "Pregnancies",
    "PlasmaGlucose",
    "DiastolicBloodPressure",
    "TricepsThickness",
    "SerumInsulin",
    "BMI",
    "DiabetesPedigree",
    "Age",
]
label_column = "Diabetic"

missing_columns = [col for col in feature_columns + [label_column] if col not in diabetes.columns]
if missing_columns:
    raise ValueError("Missing required columns: " + ", ".join(missing_columns))

X = diabetes[feature_columns].values
y = diabetes[label_column].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=0,
    stratify=y if len(np.unique(y)) > 1 else None,
)

print("Training a decision tree model...")
model = DecisionTreeClassifier(random_state=0)
model.fit(X_train, y_train)

y_hat = model.predict(X_test)
accuracy = float(np.average(y_hat == y_test))

y_scores = model.predict_proba(X_test)[:, 1]
auc = float(roc_auc_score(y_test, y_scores))

print("Accuracy:", accuracy)
print("AUC:", auc)

# MLflow metrics are useful, but the job should not fail just because tracking has a package issue.
try:
    import mlflow
    mlflow.log_metric("Accuracy", accuracy)
    mlflow.log_metric("AUC", auc)
    print("MLflow metrics logged.")
except Exception as ex:
    print("WARNING: MLflow metric logging failed, but training will continue.")
    print(type(ex).__name__, str(ex))

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
fig = plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k--")
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Decision Tree")

roc_path = output_folder / "roc_curve_decision_tree.png"
fig.savefig(roc_path, bbox_inches="tight")
plt.close(fig)
print("ROC chart saved to:", roc_path)

model_path = output_folder / "model.pkl"
joblib.dump(value=model, filename=model_path)
print("Model saved to:", model_path)

metrics_path = output_folder / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({"Accuracy": accuracy, "AUC": auc}, f, indent=2)
print("Metrics saved to:", metrics_path)

## 6. Create or reuse compute

The pipeline will run on an Azure ML compute cluster named `aml-cluster`. If it does not exist, this cell creates it.

In [ ]:
from azure.ai.ml.entities import AmlCompute

cluster_name = "aml-cluster"

try:
    compute = ml_client.compute.get(cluster_name)
    print(f"Found existing compute cluster: {cluster_name}")
    print("Provisioning state:", compute.provisioning_state)
except Exception:
    print(f"Compute cluster {cluster_name} not found. Creating it now...")
    compute = AmlCompute(
        name=cluster_name,
        size="STANDARD_D2_V2",
        min_instances=0,
        max_instances=4,
        idle_time_before_scale_down=120,
    )
    poller = ml_client.compute.begin_create_or_update(compute)
    compute = poller.result()
    print("Compute cluster created.")

print("Compute name:", compute.name)
print("Compute type:", compute.type)
print("Provisioning state:", compute.provisioning_state)

## 7. Create the environment

This replaces the old SDK v1 `RunConfiguration` / `CondaDependencies` pattern.

In [ ]:
%%writefile diabetes_pipeline_v2/conda.yml
name: diabetes-pipeline-env
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - pandas
      - numpy
      - scikit-learn
      - matplotlib
      - joblib
      - mlflow<3
      - azureml-mlflow

In [ ]:
from azure.ai.ml.entities import Environment

pipeline_env = Environment(
    name="diabetes-pipeline-env",
    description="Environment for diabetes SDK v2 pipeline lab",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file=str(experiment_folder / "conda.yml"),
)

pipeline_env = ml_client.environments.create_or_update(pipeline_env)
env_ref = f"{pipeline_env.name}:{pipeline_env.version}"

print("Environment registered:", env_ref)

## 8. Define the command component

SDK v1 used `EstimatorStep`. SDK v2 uses command components.

In [ ]:
from azure.ai.ml import Input, Output, command
from azure.ai.ml.constants import AssetTypes

train_component = command(
    name="train_diabetes_decision_tree_component",
    display_name="Train Diabetes Decision Tree",
    description="Train a decision-tree model from diabetes CSV files.",
    code=str(experiment_folder),
    command=(
        "python train_diabetes.py "
        "--training_data ${{inputs.training_data}} "
        "--output_model ${{outputs.output_model}}"
    ),
    inputs={
        "training_data": Input(type=AssetTypes.URI_FOLDER),
    },
    outputs={
        "output_model": Output(type=AssetTypes.URI_FOLDER),
    },
    environment=env_ref,
)

print("Command component defined.")

## 9. Build the pipeline

SDK v1 used `Pipeline(workspace=..., steps=...)`. SDK v2 uses the `@dsl.pipeline` decorator.

In [ ]:
from azure.ai.ml import dsl

@dsl.pipeline(
    name="diabetes_training_pipeline_v2",
    description="SDK v2 pipeline that trains a diabetes decision-tree model.",
)
def diabetes_training_pipeline(training_data):
    train_job = train_component(training_data=training_data)
    return {
        "trained_model": train_job.outputs.output_model
    }

print("Pipeline function created.")

## 10. Submit the pipeline job

The pipeline input points to the registered data asset.

In [ ]:
pipeline_input = Input(
    type=AssetTypes.URI_FOLDER,
    path=f"azureml:{diabetes_data.name}:{diabetes_data.version}",
)

pipeline_job = diabetes_training_pipeline(training_data=pipeline_input)
pipeline_job.settings.default_compute = cluster_name

returned_pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="diabetes-training-pipeline-v2"
)

print("Pipeline submitted.")
print("Job name:", returned_pipeline_job.name)
print("Status:", returned_pipeline_job.status)
print("Studio URL:", returned_pipeline_job.studio_url)

## 11. Stream logs and wait for completion

Do not skip this cell if you want to register the model from the pipeline output later.

In [ ]:
ml_client.jobs.stream(returned_pipeline_job.name)

completed_pipeline_job = ml_client.jobs.get(returned_pipeline_job.name)
print("Status:", completed_pipeline_job.status)
print("Studio URL:", completed_pipeline_job.studio_url)

if completed_pipeline_job.status != "Completed":
    raise RuntimeError(
        f"Pipeline did not complete successfully. Status: {completed_pipeline_job.status}. "
        "Open the Studio URL above and check the job logs."
    )

## 12. Register the model from the pipeline output

The old lab registered the model inside a second pipeline step using SDK v1. In SDK v2, the cleaner beginner-friendly approach is:

1. Run the pipeline.
2. Use the completed pipeline output URI.
3. Register that output as a model asset.

This avoids authentication problems inside the compute job.

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_path = f"azureml://jobs/{completed_pipeline_job.name}/outputs/trained_model"

registered_model = Model(
    name="diabetes_model",
    path=model_path,
    type=AssetTypes.CUSTOM_MODEL,
    description="Decision-tree diabetes model trained by SDK v2 pipeline",
    tags={"Training context": "SDK v2 pipeline", "lab": "06A"},
)

registered_model = ml_client.models.create_or_update(registered_model)

print("Model registered.")
print("Name:", registered_model.name)
print("Version:", registered_model.version)
print("Path:", registered_model.path)

## 13. List registered diabetes models

In [ ]:
models = list(ml_client.models.list(name="diabetes_model"))

if not models:
    print("No diabetes_model assets found.")
else:
    for model in models:
        print("Name:", model.name)
        print("Version:", model.version)
        print("Type:", model.type)
        print("Tags:", model.tags)
        print("---")

## What changed from the old lab?

| Old SDK v1 object | SDK v2 replacement |
|---|---|
| `Workspace.from_config()` | `MLClient.from_config()` |
| `Dataset.Tabular` | `Data` asset with `uri_folder` |
| `EstimatorStep` | `command` component |
| `PythonScriptStep` | usually another `command` component, or register after pipeline completion |
| `PipelineData` | `Output(type=AssetTypes.URI_FOLDER)` |
| `RunConfiguration` | `Environment` entity |
| `Experiment.submit()` | `ml_client.jobs.create_or_update()` |
| `RunDetails` widget | Studio URL + `ml_client.jobs.stream()` |
| `Model.register()` | `ml_client.models.create_or_update(Model(...))` |

The biggest design change: this version registers the model after the pipeline completes, instead of doing registration from inside a pipeline step. That is more reliable for beginner labs because it avoids compute-side workspace authentication issues.